## Numpy analysis

In this notebook I use numpy to analyse customer spending profiles from the customer category matrix

### Objectives

- Convert the pandas dataframe into a numpy array
- Practise indexing, slicing and operations by axis
- Apply boolean masks, percentiles and vectorised calculations
- Normalise customer spending profiles using broadcasting
- Identify dominant categories and spending patterns
- Validate and save the final results

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd

In [3]:
current_directory = Path.cwd()

if current_directory.name == "notebooks":
    project_root = current_directory.parent
else:
    project_root = current_directory

In [4]:
matrix_path = (
    project_root
    / "data"
    / "processed"
    / "customer_category_matrix.csv"
)

customer_category_matrix = pd.read_csv(
    matrix_path,
    index_col="customer_id",
)

customer_category_matrix.head()

,Books,Electronics,Entertainment,Fashion,Groceries,Health,Home,Office,Restaurants,Sports,Travel
customer_id,,,,,,,,,,,
C001,19.99,899.00,70.49,76.0,34.8,0.0,0.00,0.00,6.8,0.0,0.0
C002,0.00,423.99,0.00,0.0,0.0,0.0,0.00,12.60,4.2,0.0,815.0
C003,43.98,39.99,0.00,0.0,0.0,0.0,0.00,0.00,0.0,52.0,0.0
C004,0.00,249.90,0.00,0.0,0.0,0.0,58.75,14.25,0.0,82.5,410.0
C005,21.50,0.00,0.00,0.0,62.4,0.0,130.60,0.00,0.0,0.0,0.0


In [5]:
customer_ids = (customer_category_matrix.index.to_numpy())
category_names = (customer_category_matrix.columns.to_numpy())
profile_matrix = (customer_category_matrix.to_numpy(dtype=float))

In [6]:
print(type(profile_matrix))
print(customer_ids[:3])
print(category_names[:3])

<class 'numpy.ndarray'>
['C001' 'C002' 'C003']
['Books' 'Electronics' 'Entertainment']


In [7]:
array_information = pd.Series(
    {
        "rows": profile_matrix.shape[0],
        "columns": profile_matrix.shape[1],
        "dimensions": profile_matrix.ndim,
        "elements" : profile_matrix.size,
        "dtype": str(profile_matrix.dtype)
    }
)

array_information

rows               12
columns            11
dimensions          2
elements          132
dtype         float64
dtype: object

In [8]:
profile_matrix.shape == (len(customer_ids), len(category_names))

True

In [9]:
first_value = profile_matrix[0, 0]
first_customer_profile = profile_matrix[0,:]
first_category_values = profile_matrix[:,0]

In [10]:
print("First value:", first_value)
print("First customer:", customer_ids[0])
print("First customer profile:", first_customer_profile)
print("First category:", category_names[0])
print("First category values:", first_category_values)

First value: 19.99
First customer: C001
First customer profile: [ 19.99 899.    70.49  76.    34.8    0.     0.     0.     6.8    0.
   0.  ]
First category: Books
First category values: [19.99  0.   43.98  0.   21.5   0.    0.    0.    0.    0.    0.    0.  ]


In [11]:
customer_totals = profile_matrix.sum(axis = 1)
category_totals = profile_matrix.sum(axis = 0)

In [12]:
print(customer_totals.shape)
print(category_totals.shape)

(12,)
(11,)


In [14]:
customer_totals_table = pd.DataFrame(
    {
        "customer_id": customer_ids,
        "total_amount": customer_totals
    }
).sort_values(
    "total_amount",
    ascending = False
)

customer_totals_table

,customer_id,total_amount
1,C002,1255.79
7,C008,1162.25
0,C001,1107.08
3,C004,815.40
8,C009,289.49
4,C005,214.50
10,C011,186.10
9,C010,177.99
5,C006,170.85
2,C003,135.97


In [15]:
category_totals_table = pd.DataFrame(
    {
        "merchant_category": category_names,
        "total_amount": category_totals,
    }
).sort_values(
    "total_amount",
    ascending=False,
)

category_totals_table

,merchant_category,total_amount
10,Travel,2325.00
1,Electronics,1701.88
3,Fashion,527.45
4,Groceries,321.85
6,Home,263.85
2,Entertainment,252.47
9,Sports,134.50
0,Books,85.47
8,Restaurants,33.00
7,Office,26.85


In [16]:
np.isclose(
    customer_totals.sum(),
    category_totals.sum()
)

np.True_

In [17]:
active_category_mask = (
    profile_matrix > 0
)

active_categories_per_customer = (
    active_category_mask.sum(axis = 1)
)

In [18]:
active_category_mask.dtype

dtype('bool')

In [19]:
pd.DataFrame(
    {
        "customer_id": customer_ids,
        "active_categories": (active_categories_per_customer)
    }
)


,customer_id,active_categories
0,C001,6
1,C002,4
2,C003,3
3,C004,5
4,C005,3
5,C006,2
6,C007,2
7,C008,3
8,C009,3
9,C010,2


In [20]:
positive_spending_values = (profile_matrix[active_category_mask])
spending_percentiles = np.percentile(positive_spending_values, [25, 50, 75, 90])

spending_percentiles

array([ 36.0975,  70.92  , 114.1875, 414.197 ])

In [21]:
large_cell_threshold = np.percentile(positive_spending_values, 75)

large_spending_mask = (profile_matrix >= large_cell_threshold)

In [24]:
print(
    "75th percentile:",
    large_cell_threshold,
)

print(
    "Large customer-category values:",
    large_spending_mask.sum(),
)

75th percentile: 114.1875
Large customer-category values: 10


In [25]:
customer_totals_column = (
    profile_matrix.sum(axis = 1, keepdims=True
    )
)

print(profile_matrix.shape)
print(customer_totals_column.shape)

(12, 11)
(12, 1)


In [29]:
normalized_profiles= np.divide(
    profile_matrix,
    customer_totals_column,
    out = np.zeros_like(
        profile_matrix,
        dtype=float
    ),
    where = customer_totals_column != 0
)

In [30]:
normalized_profiles[0]

array([0.01805651, 0.8120461 , 0.063672  , 0.06864906, 0.03143404,
       0.        , 0.        , 0.        , 0.00614228, 0.        ,
       0.        ])

In [31]:
normalized_profiles.sum(axis=1)

array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.])

In [32]:
dominant_category_positions = (normalized_profiles.argmax(axis = 1))
dominant_categories = (category_names[dominant_category_positions])
dominant_shares = (normalized_profiles.max(axis = 1))

In [33]:
pd.DataFrame(
    {
        "customer_id": customer_ids,
        "dominant_category": (
            dominant_categories
        ),
        "dominant_share": (
            dominant_shares
        ),
    }
)

,customer_id,dominant_category,dominant_share
0,C001,Electronics,0.812046
1,C002,Travel,0.648994
2,C003,Sports,0.382437
3,C004,Travel,0.502821
4,C005,Home,0.608858
5,C006,Electronics,0.520925
6,C007,Fashion,0.872299
7,C008,Travel,0.813078
8,C009,Travel,0.535424
9,C010,Fashion,0.674195


In [34]:
high_spend_threshold = np.percentile(
    customer_totals,
    75,
)

spending_level = np.where(
    customer_totals
    >= high_spend_threshold,
    "High spender",
    "Regular spender",
)

In [35]:
pd.DataFrame(
    {
        "customer_id": customer_ids,
        "total_amount": customer_totals,
        "spending_level": spending_level,
    }
)

,customer_id,total_amount,spending_level
0,C001,1107.08,High spender
1,C002,1255.79,High spender
2,C003,135.97,Regular spender
3,C004,815.40,Regular spender
4,C005,214.50,Regular spender
5,C006,170.85,Regular spender
6,C007,101.80,Regular spender
7,C008,1162.25,High spender
8,C009,289.49,Regular spender
9,C010,177.99,Regular spender


In [36]:
concentration_conditions = [
    dominant_shares >= 0.70,
    dominant_shares >= 0.40,
]

concentration_choices = [
    "Highly concentrated",
    "Moderately concentrated",
]

profile_concentration = np.select(
    concentration_conditions,
    concentration_choices,
    default="Diversified",
)

In [37]:
pd.DataFrame(
    {
        "customer_id": customer_ids,
        "dominant_share": dominant_shares,
        "profile_concentration": (
            profile_concentration
        ),
    }
)

,customer_id,dominant_share,profile_concentration
0,C001,0.812046,Highly concentrated
1,C002,0.648994,Moderately concentrated
2,C003,0.382437,Diversified
3,C004,0.502821,Moderately concentrated
4,C005,0.608858,Moderately concentrated
5,C006,0.520925,Moderately concentrated
6,C007,0.872299,Highly concentrated
7,C008,0.813078,Highly concentrated
8,C009,0.535424,Moderately concentrated
9,C010,0.674195,Moderately concentrated


In [38]:
median_customer_total = np.median(
    customer_totals
)

relative_spend_index = (
    customer_totals / median_customer_total
)

capped_spend_index = np.clip(
    relative_spend_index,
    0,
    3,
)

In [39]:
print(relative_spend_index)
print(capped_spend_index)

[5.52710934 6.26954568 0.67883175 4.07089366 1.07089366 0.85297054
 0.50823764 5.80254618 1.44528208 0.88861707 0.92910634 0.36445332]
[3.         3.         0.67883175 3.         1.07089366 0.85297054
 0.50823764 3.         1.44528208 0.88861707 0.92910634 0.36445332]


In [40]:
# Final summary
customer_numpy_summary = pd.DataFrame(
    {
        "customer_id": customer_ids,
        "total_completed_purchase_amount": (
            customer_totals
        ),
        "active_categories": (
            active_categories_per_customer
        ),
        "dominant_category": (
            dominant_categories
        ),
        "dominant_category_share": (
            dominant_shares
        ),
        "spending_level": (
            spending_level
        ),
        "profile_concentration": (
            profile_concentration
        ),
        "relative_spend_index": (
            relative_spend_index
        ),
        "capped_spend_index": (
            capped_spend_index
        ),
    }
)

In [41]:
customer_numpy_summary[
    [
        "total_completed_purchase_amount",
        "dominant_category_share",
        "relative_spend_index",
        "capped_spend_index",
    ]
] = (
    customer_numpy_summary[
        [
            "total_completed_purchase_amount",
            "dominant_category_share",
            "relative_spend_index",
            "capped_spend_index",
        ]
    ]
    .round(3)
)

In [42]:
customer_numpy_summary = (
    customer_numpy_summary
    .sort_values(
        "total_completed_purchase_amount",
        ascending=False,
    )
    .reset_index(drop=True)
)

customer_numpy_summary

,customer_id,total_completed_purchase_amount,active_categories,dominant_category,dominant_category_share,spending_level,profile_concentration,relative_spend_index,capped_spend_index
0,C002,1255.79,4,Travel,0.649,High spender,Moderately concentrated,6.270,3.000
1,C008,1162.25,3,Travel,0.813,High spender,Highly concentrated,5.803,3.000
2,C001,1107.08,6,Electronics,0.812,High spender,Highly concentrated,5.527,3.000
3,C004,815.40,5,Travel,0.503,Regular spender,Moderately concentrated,4.071,3.000
4,C009,289.49,3,Travel,0.535,Regular spender,Moderately concentrated,1.445,1.445
5,C005,214.50,3,Home,0.609,Regular spender,Moderately concentrated,1.071,1.071
6,C011,186.10,3,Fashion,0.520,Regular spender,Moderately concentrated,0.929,0.929
7,C010,177.99,2,Fashion,0.674,Regular spender,Moderately concentrated,0.889,0.889
8,C006,170.85,2,Electronics,0.521,Regular spender,Moderately concentrated,0.853,0.853
9,C003,135.97,3,Sports,0.382,Regular spender,Diversified,0.679,0.679


In [43]:
normalized_profiles_df = pd.DataFrame(
    normalized_profiles,
    index=customer_ids,
    columns = category_names
)

normalized_profiles_df.index.name = ("customer_id")

normalized_profiles_df.head()

,Books,Electronics,Entertainment,Fashion,Groceries,Health,Home,Office,Restaurants,Sports,Travel
customer_id,,,,,,,,,,,
C001,0.018057,0.812046,0.063672,0.068649,0.031434,0.0,0.000000,0.000000,0.006142,0.000000,0.000000
C002,0.000000,0.337628,0.000000,0.000000,0.000000,0.0,0.000000,0.010034,0.003345,0.000000,0.648994
C003,0.323454,0.294109,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.382437,0.000000
C004,0.000000,0.306475,0.000000,0.000000,0.000000,0.0,0.072051,0.017476,0.000000,0.101177,0.502821
C005,0.100233,0.000000,0.000000,0.000000,0.290909,0.0,0.608858,0.000000,0.000000,0.000000,0.000000


In [45]:
expected_profile_sums = np.where(
    customer_totals > 0,
    1.0,
    0.0,
)

In [47]:
numpy_validation = pd.Series(
    {
        "array_shape_matches_dataframe": (
            profile_matrix.shape
            == customer_category_matrix.shape
        ),
        "all_values_are_non_negative": (
            np.all(profile_matrix >= 0)
        ),
        "customer_and_category_totals_match": (
            np.isclose(
                customer_totals.sum(),
                category_totals.sum(),
            )
        ),
        "normalised_rows_sum_correctly": (
            np.isclose(
                normalized_profiles.sum(axis=1),
                expected_profile_sums,
                atol=1e-10,
            ).all()
        ),
        "normalised_profiles_have_no_nan": (
            ~np.isnan(
                normalized_profiles
            ).any()
        ),
        "dominant_shares_are_valid": (
            np.all(
                (dominant_shares >= 0)
                & (dominant_shares <= 1)
            )
        ),
        "summary_has_one_row_per_customer": (
            len(customer_numpy_summary)
            == len(customer_ids)
        ),
    },
    name="passed",
)

numpy_validation

array_shape_matches_dataframe         True
all_values_are_non_negative           True
customer_and_category_totals_match    True
normalised_rows_sum_correctly         True
normalised_profiles_have_no_nan       True
dominant_shares_are_valid             True
summary_has_one_row_per_customer      True
Name: passed, dtype: bool

In [48]:
# Save results

normalized_profiles_path = (
    project_root
    / "data"
    / "processed"
    / "customer_profiles_normalized.csv"
)

numpy_summary_path = (
    project_root
    / "data"
    / "processed"
    / "customer_numpy_summary.csv"
)

In [49]:
normalized_profiles_df.to_csv(
    normalized_profiles_path
)

customer_numpy_summary.to_csv(
    numpy_summary_path,
    index=False,
)

## Numpy analysis results

The customer spending matrix was converted from pandas into a numpy array. This phase applied:

- Array attributes and indexing
- Row wise and column wise operations
- Boolean masks
- Vectorised calculations
- Percentiles
- Broadcasting
- Safe array division
- Profile normalisation

The resulting normalised profiles describe the proportion of each customer's spending assigned to every merchant category. Later these profiles will support customer comparison and anomaly detection